# Kahneman Framing × TRIBE v2

**No HF_TOKEN. No secrets. No git clone.**

1. Runtime → **T4 GPU**
2. Runtime → **Run all** (twice if the notebook restarts after installing dependencies)

> Open only this link (not a saved Drive copy):
> https://colab.research.google.com/github/akifnu/DSprojects/blob/main/tribev2/notebooks/Framing_RCT_NoSetup.ipynb

In [ ]:
# VERSION token-free-2026-06-16 — must run first
import os, shutil, sys
NOTEBOOK_VERSION = 'token-free-2026-06-16'
shutil.rmtree('/content/DSprojects', ignore_errors=True)
for _k in ('HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN'):
    os.environ.pop(_k, None)
print(f'OK: {NOTEBOOK_VERSION} | no secrets | no clone')

In [ ]:
# Colab ships a NumPy version that breaks tribev2/neuralset — pin 2.2.6 first
import subprocess, sys
from pathlib import Path

MARKER = Path('/content/.tribev2_deps_installed')

if not MARKER.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'], check=False)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==2.2.6'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'tribev2 @ git+https://github.com/facebookresearch/TRIBEv2.git',
        'gtts', 'scipy', 'pandas', 'scikit-learn',
    ])
    MARKER.write_text('ok')
    print('Installed numpy==2.2.6 + tribev2.')
    print('Restarting runtime once (required) — then click Runtime → Run all again.')
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)
else:
    import numpy as np
    from tribev2 import TribeModel
    assert np.__version__ == '2.2.6', np.__version__
    print(f'Ready: numpy {np.__version__} | tribev2 imported')

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
FRAMING_PAIRS = [
  {"id": "asian_disease", "domain": "health",
   "gain": "Program A will save 200 people for certain. Program B has a one-third probability that all 600 people will be saved.",
   "loss": "Program C will result in 400 people dying for certain. Program D has a one-third probability that nobody will die."},
  {"id": "surgery", "domain": "health",
   "gain": "The operation has a 90 percent success rate. Nine out of ten patients recover fully.",
   "loss": "The operation has a 10 percent failure rate. One out of ten patients do not survive."},
  {"id": "money_wallet", "domain": "financial",
   "gain": "You can keep 20 dollars for sure, or gamble fifty-fifty to keep 30 dollars or only 10 dollars.",
   "loss": "You will lose 10 dollars for sure and keep 20, or gamble fifty-fifty to lose nothing and keep 30, or lose 20 and keep 10."},
  {"id": "credit_card", "domain": "financial",
   "gain": "Paying cash gives you a 1 dollar discount compared to the credit card price.",
   "loss": "Paying by credit card adds a 1 dollar surcharge compared to the cash price."},
  {"id": "employment", "domain": "economic",
   "gain": "The new policy will help 80 percent of workers keep their jobs next year.",
   "loss": "The new policy means 20 percent of workers will lose their jobs next year."},
  {"id": "beef", "domain": "consumer",
   "gain": "The label says the ground beef is 75 percent lean.",
   "loss": "The label says the ground beef is 25 percent fat."},
  {"id": "exam", "domain": "education",
   "gain": "You answered 60 percent of the exam questions correctly.",
   "loss": "You answered 40 percent of the exam questions incorrectly."},
  {"id": "vaccine", "domain": "health",
   "gain": "The vaccine caused no serious side effects in 95 percent of recipients.",
   "loss": "The vaccine caused mild side effects in 5 percent of recipients."},
  {"id": "treatment_85", "domain": "health",
   "gain": "This treatment works for 85 percent of patients.",
   "loss": "This treatment fails for 15 percent of patients."},
  {"id": "investment", "domain": "financial",
   "gain": "The fund gained value on 70 percent of trading days last year.",
   "loss": "The fund lost value on 30 percent of trading days last year."},
  {"id": "pollution", "domain": "environment",
   "gain": "The cleanup plan removes 40 percent of river pollution within five years.",
   "loss": "The cleanup plan leaves 60 percent of river pollution in place within five years."},
  {"id": "course_pass", "domain": "education",
   "gain": "72 percent of students passed the certification course on the first attempt.",
   "loss": "28 percent of students failed the certification course on the first attempt."},
]
print(len(FRAMING_PAIRS), 'pairs loaded')

In [ ]:
from pathlib import Path
from gtts import gTTS

AUDIO_DIR = Path('/content/framing_audio')
AUDIO_DIR.mkdir(exist_ok=True)
for pair in FRAMING_PAIRS:
    for frame in ('gain', 'loss'):
        path = AUDIO_DIR / f"{pair['id']}_{frame}.mp3"
        if not path.exists():
            gTTS(pair[frame], lang='en').save(str(path))
print('audio files:', len(list(AUDIO_DIR.glob('*.mp3'))))

In [ ]:
import numpy as np
from tribev2 import TribeModel

model = TribeModel.from_pretrained('facebook/tribev2', cache_folder='/content/tribe_cache', device='cuda')
print('TRIBE v2 loaded on GPU')

In [ ]:
def predict_audio(path):
    events = model.get_events_dataframe(audio_path=str(path))
    preds, _ = model.predict(events=events, verbose=False)
    return np.asarray(preds)

results = []
for pair in FRAMING_PAIRS:
    row = {'id': pair['id'], 'domain': pair['domain']}
    for frame in ('gain', 'loss'):
        preds = predict_audio(AUDIO_DIR / f"{pair['id']}_{frame}.mp3")
        row[f'{frame}_mean_abs'] = float(np.mean(np.abs(preds)))
    row['loss_minus_gain'] = row['loss_mean_abs'] - row['gain_mean_abs']
    results.append(row)
    print(pair['id'], f"{row['loss_minus_gain']:+.4f}")

In [ ]:
import pandas as pd
from scipy import stats

df = pd.DataFrame(results)
display(df)
diff = df['loss_mean_abs'].values - df['gain_mean_abs'].values
_, p = stats.ttest_rel(df['loss_mean_abs'], df['gain_mean_abs'])
print(f"loss>gain: {(diff>0).sum()}/{len(df)}  mean_diff={diff.mean():.4f}  p={p:.4f}")